# Cross integration: Multiple RNA + ATAC

Cross integration combines batches that all measure the same modalities. The task is to remove batch effects and keep the biological structure. This tutorial is for batches that each hold RNA and ATAC peaks. It runs StabMap and scMM on `D56mini`, 4,500 cells in three batches, cut from the first three batches of the benchmark dataset `D56`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_cross_rna_atac.ipynb).

## 1. Install

This cell installs `multibench-sc`. It keeps the numpy and pandas that are already installed, so Colab needs no restart.

<details>
<summary>Details</summary>

On Colab, choose a GPU runtime before you run the notebook: Runtime -> Change runtime type -> T4 GPU. On a CPU runtime, an environment that has a smaller CPU build gets that build, and training methods run slower.

</details>

In [ ]:
import numpy, pandas
%pip install -q multibench-sc numpy=={numpy.__version__} pandas=={pandas.__version__}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D56mini` (10 MB) once. Each method runs in its own environment. `mtb.env.install` downloads the environments for StabMap and scMM, with no conda needed. The download is 1.4 GB on a computer without a GPU and 2.0 GB on one with a GPU.

<details>
<summary>Details</summary>

`state` is `PACKED` for an environment downloaded now and `have` for one that was already there. Without `dry_run=False`, `mtb.env.install` downloads nothing and returns the plan with its sizes.

Environments go to `mtb.config.DEFAULT.envs_dir`. To use another disk, set it before this cell.

From a terminal, `multibench env install --methods StabMap,scMM --packed --run` does the same.

</details>

In [ ]:
import pandas as pd
import multibench as mtb

mtb.data.fetch("D56mini")

In [ ]:
METHODS = ["StabMap", "scMM"]
envs = mtb.env.install(METHODS, dry_run=False)
pd.DataFrame(envs)[["env", "methods", "state"]]

## 3. Run the methods

`run_all` runs each method on `D56mini` and scores its output with the scIB metrics. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

Each method writes an embedding: a table of numbers with one row per cell. `run_all` scores it against the cell-type labels of `D56mini`.

The clustering metrics `ARI`, `NMI`, `ASW`, `iASW`, `iF1` and `cLISI` measure how well the embedding separates the cell types. The batch metrics `ASW_batch`, `GC` and `iLISI` measure how well the batches mix. They appear only when the data has several batches.

`res.failures` says why a method failed. `params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_batch("out/D56mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run_all("D56mini", "cross", methods=METHODS,
                  out_dir="out/D56mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size shows the rank within a column, and bigger is better. The fill compares a value with the other rows in the same column.

<details>
<summary>Details</summary>

The lightest fill is the lowest value in this figure, not zero.

Metrics are grouped by family: blue for dimension reduction and clustering, green for batch correction. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and the note under the figure names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts and an ATAC peak matrix of the same cells, with a cell type and a batch for each cell. `mtb.io.export_dataset` with `batch=` writes one set of files per batch, and `run_all` runs on that folder. Give the same peaks in every batch.

<details>
<summary>Details: export</summary>

For one AnnData per batch, call `export_dataset` once per batch with `batch_index=N` instead of `batch=`.

The folder name, `MYCROSS`, is the dataset name for `run_all`, and `data_path` is the folder that holds it.

`mtb.scan("MYCROSS", "cross", data_path="mydata")` checks the folder and the environments without running anything. Its `reason` column says what is missing.

`overwrite=True` replaces the files of an earlier run of this cell. Without it, `export_dataset` raises `FileExistsError` rather than replace a file.

</details>

Here 60% of `D56mini`'s cells, as an RNA AnnData with a batch column and a peak AnnData, take the place of your data:

In [ ]:
import anndata as ad
import scanpy as sc

d = mtb.config.DEFAULT.data_path / "D56mini"
rna, atac = [], []
for b in (1, 2, 3):
    r = mtb.io.read_canonical(d / f"rna{b}.h5")
    r.obs["celltype"] = pd.read_csv(d / f"cty{b}.csv")["x"].values
    rna.append(r)
    atac.append(mtb.io.read_canonical(d / f"atac{b}.h5"))
rna = ad.concat(rna, label="batch", keys=["1", "2", "3"], index_unique="-")
atac = ad.concat(atac, keys=["1", "2", "3"], index_unique="-")
rna = sc.pp.subsample(rna, fraction=0.6, random_state=0, copy=True)
atac = atac[rna.obs_names].copy()
rna, atac

Write the folder, then run the same methods on it:

In [ ]:
mtb.io.export_dataset(rna, "mydata/MYCROSS", atac=atac, atac_kind="peak",
                      labels="obs:celltype", batch="obs:batch", category="cross",
                      overwrite=True)

In [ ]:
mine = mtb.run_all("MYCROSS", "cross", methods=METHODS, data_path="mydata",
                   out_dir="out/MYCROSS")
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

8 methods have a variant for cross Multiple RNA + ATAC: MOFA2, Multigrate, StabMap, UINMF, UnitedNet, scMDC, scMM and scMoMaT. Each runs on `D56mini`. To run them all, set `METHODS` to that list in section 2 and run the notebook again. That downloads 7 envs, 27.6 GB to download on a CPU host, 31.9 GB on a GPU host.

UINMF's script takes two batches, so it reads batches 1 and 2.

[Every method on `D56mini`](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_atac_all/) shows that run. UnitedNet and scMDC run only with one more `run_all` argument, which that page shows.

## Troubleshooting

`res.failures` lists each method that failed or was skipped. Its `error` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| `env.install` refuses on macOS or Windows | methods run only on Linux: use Colab or a Linux machine |
| a method needs an NVIDIA GPU | choose a GPU runtime on Colab, or a machine with a GPU |
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run_all` |
| low `label_order_confidence` | several label files fit the cell count: check `label_order_candidates` in `res.results` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- `mtb.cite(METHODS)` returns the citations for the benchmark and the methods you ran.
- The other tutorials: [vertical RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt/), [vertical RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_atac/), [vertical RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt_atac/), [diagonal [RNA, ATAC]](https://dsichang.github.io/scMultiBench/tutorials/diagonal_rna_atac/), [diagonal [multiple RNA, multiple ATAC]](https://dsichang.github.io/scMultiBench/tutorials/diagonal_multi/), [mosaic [RNA, RNA + ADT, ADT]](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_adt/), [mosaic [RNA, RNA + ATAC, ATAC]](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_atac/), [mosaic Mixed, with shared modality](https://dsichang.github.io/scMultiBench/tutorials/mosaic_shared/), [mosaic Mixed, without shared modality](https://dsichang.github.io/scMultiBench/tutorials/mosaic_unshared/), [cross Multiple RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_adt/), [cross Multiple ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/cross_adt_atac/), [cross Multiple RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_adt_atac/).
- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.